# HW 6 — H5 on val

**Pass criteria (Oct 25, 2026):**
- Primary: Full main breaker test (cutoff 0.9, tau 1 s): (obs − exp)/n > 0
  and the 95% group-bootstrap CI excludes 0.
- Secondary: confound subsets (a) and (b): point estimates > 0.
- Secondary: on Full, k = 1 has the longest KM median stall.
- Descriptive: Full mean % committed, compared with train (9.7%).

Val groups only. `h5.run` is the frozen analysis from `h5_obdurate.ipynb` (`src/slider/h5.py` unchanged); it is run once here. Train numbers in the summary come from `results/tables/h5_train_checks.csv`. All val tables are saved to `results/tables/h5_val.csv`.

In [1]:
import pandas as pd
from slider import config
from slider import h5

res = h5.run(roles={"val"})
res["checks"].to_csv(config.H5_VAL, index=False)
print(len(res["checks"]), "rows saved to", config.H5_VAL.name)

21 rows saved to h5_val.csv


## Main setting (cutoff 0.9, τ = 1 s)

In [2]:
res["main"].round(2)

,runs,mean % committed,spells,KM median (s),consensus %,relapse %,unresolved %,censored %
topology,,,,,,,,
Full,26,8.65,63,2.40,23.81,73.02,3.17,0.00
Hierarchy,44,13.07,189,2.00,13.23,84.66,0.53,1.59
Star,6,16.36,13,1.54,38.46,61.54,0.00,0.00


Full by k = minority size at the spell's last committed row.

In [3]:
res["full_by_k"].round(2)

,spells,KM median (s),consensus %,relapse %,unresolved %,censored %
k,,,,,,
1,37,4.10,27.03,70.27,2.70,0.0
2,15,1.30,20.00,73.33,6.67,0.0
3,11,0.99,18.18,81.82,0.00,0.0


Breaker test (Full): null P(minority) = k/7; 95% CI from 10,000 resamples of groups, seed 573.

In [4]:
pd.Series(res["breaker"]).round(3).to_frame("Full")

,Full
spells ended,63.000
ambiguous,1.000
no drop,0.000
breaker spells (n),62.000
groups,9.000
observed minority,20.000
expected minority,14.000
observed share,0.323
expected share,0.226
(obs - exp) / n,0.097


Was the breaker the last player to reach |s| ≥ 0.9? (null 1/7)

In [5]:
pd.Series(res["confound"]).round(3).to_frame("Full")

,Full
breaker spells with a clear last reacher (n),62.000
breaker = last to reach,19.000
share,0.306
null share,0.143
ambiguous last reacher,0.000


## Checks

### Confound subsets

In [6]:
res["checks"][res["checks"]["check"] == "confound"].dropna(axis=1, how="all").round(3)

,check,variant,part,key,breaker spells (n),groups,observed minority,expected minority,observed share,expected share,(obs - exp) / n,CI low,CI high
7,confound,"(a) breaker is not the last arriver, expected ...",breaker,Full,43.0,9.0,11.0,9.667,0.256,0.225,0.031,-0.061,0.135
8,confound,"(b) last arriver in the majority, expected k/7",breaker,Full,42.0,8.0,10.0,7.857,0.238,0.187,0.051,-0.028,0.166


### Minimum spell duration (Full)

In [7]:
res["checks"][res["checks"]["check"] == "min duration"].dropna(axis=1, how="all").round(3)

,check,variant,part,key,spells,KM median (s),consensus %,relapse %,unresolved %,censored %,...,no drop,breaker spells (n),groups,observed minority,expected minority,observed share,expected share,(obs - exp) / n,CI low,CI high
9,min duration,spells >= 0.5 s,Full by k,1,35.0,4.647,28.571,68.571,2.857,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,min duration,spells >= 0.5 s,Full by k,2,14.0,1.300,21.429,71.429,7.143,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,min duration,spells >= 0.5 s,Full by k,3,9.0,1.036,11.111,88.889,0.000,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12,min duration,spells >= 0.5 s,breaker,Full,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,57.0,9.0,18.0,12.571,0.316,0.221,0.095,-0.008,0.230
13,min duration,spells >= 1.0 s,Full by k,1,33.0,5.100,27.273,69.697,3.030,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
14,min duration,spells >= 1.0 s,Full by k,2,10.0,1.600,10.000,80.000,10.000,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15,min duration,spells >= 1.0 s,Full by k,3,5.0,1.200,20.000,80.000,0.000,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
16,min duration,spells >= 1.0 s,breaker,Full,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,47.0,8.0,15.0,9.429,0.319,0.201,0.119,0.029,0.261


### Without whole-second files

In [8]:
res["checks"][res["checks"]["check"] == "no whole-second files"].dropna(axis=1, how="all").round(3)

,check,variant,part,key,runs,mean % committed,spells,KM median (s),consensus %,relapse %,...,no drop,breaker spells (n),groups,observed minority,expected minority,observed share,expected share,(obs - exp) / n,CI low,CI high
17,no whole-second files,coarse_time files dropped,topology,Full,26.0,8.646,63.0,2.400,23.810,73.016,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18,no whole-second files,coarse_time files dropped,topology,Hierarchy,44.0,13.068,189.0,2.000,13.228,84.656,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19,no whole-second files,coarse_time files dropped,topology,Star,6.0,16.358,13.0,1.544,38.462,61.538,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,no whole-second files,coarse_time files dropped,breaker,Full,NaN,NaN,NaN,NaN,NaN,NaN,...,0.0,62.0,9.0,20.0,14.0,0.323,0.226,0.097,0.004,0.222


## Summary: train vs val (pass? is for val)

In [9]:
train = pd.read_csv(config.H5_TRAIN_CHECKS)
val = res["checks"]

def pick(checks, check, part, key, variant=""):
    d = checks[(checks["check"] == check) & (checks["part"] == part) & (checks["key"].astype(str) == str(key))
               & checks["variant"].str.startswith(variant)]
    assert len(d) == 1
    return d.iloc[0]

def ci(r):
    return f'{r["(obs - exp) / n"]:.3f} [{r["CI low"]:.3f}, {r["CI high"]:.3f}]'

rows = []
t, v = pick(train, "main", "breaker", "Full"), pick(val, "main", "breaker", "Full")
rows.append(["Primary: Full breaker test, (obs - exp)/n > 0 and CI excludes 0", ci(t), ci(v),
             v["(obs - exp) / n"] > 0 and v["CI low"] > 0])
for sub in ["(a)", "(b)"]:
    t, v = pick(train, "confound", "breaker", "Full", sub), pick(val, "confound", "breaker", "Full", sub)
    rows.append([f"Secondary: confound {sub} point estimate > 0", ci(t), ci(v), v["(obs - exp) / n"] > 0])
medians = {}
for name, c in [("train", train), ("val", val)]:
    medians[name] = [pick(c, "main", "Full by k", k)["KM median (s)"] for k in [1, 2, 3]]
text = {name: " / ".join(f"k={k}: {m:.2f}" for k, m in zip([1, 2, 3], ms)) for name, ms in medians.items()}
rows.append(["Secondary: Full, k = 1 has the longest KM median (s)", text["train"], text["val"],
             medians["val"][0] > max(medians["val"][1:])])
t, v = pick(train, "main", "topology", "Full"), pick(val, "main", "topology", "Full")
rows.append(["Descriptive: Full mean % committed", f'{t["mean % committed"]:.2f}', f'{v["mean % committed"]:.2f}', "—"])

summary = pd.DataFrame(rows, columns=["criterion", "train", "val", "pass?"])
summary

,criterion,train,val,pass?
0,"Primary: Full breaker test, (obs - exp)/n > 0 ...","0.138 [0.075, 0.203]","0.097 [0.004, 0.222]",True
1,Secondary: confound (a) point estimate > 0,"0.092 [0.021, 0.159]","0.031 [-0.061, 0.135]",True
2,Secondary: confound (b) point estimate > 0,"0.105 [0.038, 0.174]","0.051 [-0.028, 0.166]",True
3,"Secondary: Full, k = 1 has the longest KM medi...",k=1: 5.00 / k=2: 1.32 / k=3: 0.94,k=1: 4.10 / k=2: 1.30 / k=3: 0.99,True
4,Descriptive: Full mean % committed,9.74,8.65,—
